# 🌦️ Weather Data Analyser — Campus Rooftop Station (Oct–Nov 2026)

**Student:** Ashish Kumar  |  **Admission No.:** 2026B15310080
**Course:** B.Tech 1st Year  |  **Level:** Beginner  |  **Tools:** Python · NumPy · pandas  |  **Github repo:** https://github.com/KODINGOBOT/Weather_Data_Analyser.git

---
### What does this notebook do?
It studies 46 days of weather readings (1 Oct – 15 Nov 2026) from a small *rooftop weather station* and answers simple questions:

* How did temperature, humidity, rain and wind change as the season moved from warm to cool?
* Which was the hottest day, the coldest night, the wettest day?
* How **comfortable** was each day? (I made my own score for this — the **Ashish Comfort Score**)
* Which week was the best / the worst?

## Step 1 — Import the libraries
* **NumPy** → fast maths on arrays (mean, std, percentile, correlation…)
* **pandas** → tables (DataFrames), cleaning, grouping

In [25]:
import numpy as np
import pandas as pd

print("Libraries loaded successfully ✅")

Libraries loaded successfully ✅


## Step 2 — Create my own unique dataset
Instead of downloading a ready-made file that everyone uses, I **generate** the readings with NumPy.

The random generator is seeded with the **digits of my admission number**, so the numbers are always the same for me (repeatable) but different from every other student's dataset.
I also add a few hand-written notes (`my_note`) for some days — you can replace them with your own.

In [26]:
# ---- 1. Unique seed made from my admission number ----
ADMISSION_NO = "2026B15310080"
seed = int("".join(ch for ch in ADMISSION_NO if ch.isdigit()))
rng = np.random.default_rng(seed)

# ---- 2. Dates: 46 days from 1 Oct 2026 ----
n_days = 46
dates = pd.date_range(start="2026-10-01", periods=n_days, freq="D")
t = np.linspace(0, 1, n_days)   # 0 = start of October, 1 = mid November (season goes warm -> cool)

# ---- 3. Weather readings (trend + random ups and downs) ----
temp_max = 33 - 9 * t + rng.normal(0, 1.4, n_days)                 # daytime high (°C)
temp_min = 21 - 11 * t + rng.normal(0, 1.2, n_days)                # night low (°C)
humidity = np.clip(70 - 18 * t + rng.normal(0, 6, n_days), 25, 95) # percent
rain_today = rng.random(n_days) < (0.30 - 0.22 * t)                # rain chance falls with time
rainfall = np.round(rain_today * rng.gamma(2.0, 4.0, n_days), 1)   # mm
wind = np.round(np.clip(rng.normal(9, 3, n_days), 1, None), 1)     # km/h

weather = pd.DataFrame({
    "date": dates,
    "temp_max_C": np.round(temp_max, 1),
    "temp_min_C": np.round(temp_min, 1),
    "humidity_pct": np.round(humidity, 0),
    "rainfall_mm": rainfall,
    "wind_kmph": wind,
})

# ---- 4. Make the data a little "messy" on purpose (real data always is) ----
missing_rows = rng.choice(n_days, size=5, replace=False)
weather.loc[missing_rows[:3], "humidity_pct"] = np.nan   # sensor failed 3 times
weather.loc[missing_rows[3:], "wind_kmph"] = np.nan      # wind meter failed 2 times

# ---- 5. My personal notes for a few days (edit these!) ----
my_notes = {
    "2026-10-05": "Lab practical day",
    "2026-10-12": "Unit test week begins",
    "2026-10-26": "Sports day (morning)",
    "2026-11-01": "Weekend trip planned",
    "2026-11-09": "Mini-project work",
}
weather["my_note"] = weather["date"].dt.strftime("%Y-%m-%d").map(my_notes).fillna("")

# ---- 6. Save as CSV and read it back (this is how real projects work) ----
weather.to_csv("ashish_weather_raw.csv", index=False)
weather = pd.read_csv("ashish_weather_raw.csv", parse_dates=["date"])
weather["my_note"] = weather["my_note"].fillna("")

print("Dataset created and saved as ashish_weather_raw.csv")
print("Rows, Columns =", weather.shape)

Dataset created and saved as ashish_weather_raw.csv
Rows, Columns = (46, 7)


## Step 3 — First look at the data
`head()`, `info()` and `describe()` are the three first commands every data analyst runs.

In [27]:
print("FIRST 8 ROWS")
print(weather.head(8).to_string(index=False))

print("\nCOLUMN INFO")
weather.info()

print("\nSUMMARY STATISTICS")
print(weather.describe().round(2).to_string())

FIRST 8 ROWS
      date  temp_max_C  temp_min_C  humidity_pct  rainfall_mm  wind_kmph           my_note
2026-10-01        33.6        20.7          67.0          0.0        2.6                  
2026-10-02        32.5        19.9          67.0          0.0       11.5                  
2026-10-03        31.2        19.8          70.0          0.0       11.4                  
2026-10-04        29.2        21.3          74.0          0.0        8.5                  
2026-10-05        31.2        19.1          81.0          3.8        8.3 Lab practical day
2026-10-06        33.5        20.7          73.0          0.0        NaN                  
2026-10-07        31.1        20.7          65.0          0.0        9.4                  
2026-10-08        33.9        20.7          68.0          4.2        5.2                  

COLUMN INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---

## Step 4 — Clean the data (fill the missing values)
Missing readings are filled using **linear interpolation** — pandas draws a straight line between the day before and the day after.

In [28]:
print("Missing values BEFORE cleaning:")
print(weather.isna().sum().to_string())

weather["humidity_pct"] = weather["humidity_pct"].interpolate(limit_direction="both").round(0)
weather["wind_kmph"] = weather["wind_kmph"].interpolate(limit_direction="both").round(1)

print("\nMissing values AFTER cleaning:", int(weather.isna().sum().sum()))

Missing values BEFORE cleaning:
date            0
temp_max_C      0
temp_min_C      0
humidity_pct    3
rainfall_mm     0
wind_kmph       2
my_note         0

Missing values AFTER cleaning: 0


## Step 5 — Create new columns (feature engineering)
Here I add:
* `temp_avg_C`, `temp_range_C` – simple maths on existing columns
* `day_name`, `week_no`, `month` – taken from the date
* `sky` and `rain_level` – words chosen with `np.select` (like a multi-way if-else)
* `comfort_score` – **my own formula** (the Ashish Comfort Score)

**Ashish Comfort Score (0–100)** starts at 100 and loses points for:
| Reason | Points lost |
|---|---|
| Average temperature away from the ideal 24 °C | 3 × difference |
| Humidity away from the ideal 50 % | 0.4 × difference |
| Rain | 1.5 × mm (maximum 20) |
| Strong wind above 20 km/h | 1 × extra km/h |

In [17]:
weather["temp_avg_C"] = ((weather["temp_max_C"] + weather["temp_min_C"]) / 2).round(1)
weather["temp_range_C"] = (weather["temp_max_C"] - weather["temp_min_C"]).round(1)
weather["day_name"] = weather["date"].dt.day_name()
weather["month"] = weather["date"].dt.month_name()
weather["week_no"] = ((weather["date"] - weather["date"].min()).dt.days // 7) + 1

# Sky condition (first matching rule wins)
weather["sky"] = np.select(
    [weather["rainfall_mm"] >= 0.1, weather["humidity_pct"] >= 70, weather["humidity_pct"] >= 55],
    ["Rainy", "Cloudy", "Partly Cloudy"],
    default="Sunny",
)

# Rain level
weather["rain_level"] = np.select(
    [weather["rainfall_mm"] < 0.1, weather["rainfall_mm"] < 5, weather["rainfall_mm"] < 15],
    ["Dry", "Light", "Moderate"],
    default="Heavy",
)

# ---- My own formula: Ashish Comfort Score ----
def comfort_score(row):
    score = 100
    score -= 3.0 * abs(row["temp_avg_C"] - 24)       # ideal temperature = 24 °C
    score -= 0.4 * abs(row["humidity_pct"] - 50)     # ideal humidity = 50 %
    score -= min(1.5 * row["rainfall_mm"], 20)       # rain spoils the day (max 20 points)
    score -= max(row["wind_kmph"] - 20, 0)           # only strong wind matters
    return max(0, min(100, round(score, 1)))

weather["comfort_score"] = weather.apply(comfort_score, axis=1)

weather["comfort_label"] = np.select(
    [weather["comfort_score"] >= 80, weather["comfort_score"] >= 65, weather["comfort_score"] >= 50],
    ["Excellent", "Good", "Fair"],
    default="Poor",
)

print(weather[["date", "temp_avg_C", "humidity_pct", "rainfall_mm", "sky", "comfort_score", "comfort_label"]].head(8).to_string(index=False))

      date  temp_avg_C  humidity_pct  rainfall_mm           sky  comfort_score comfort_label
2026-10-01        27.2          67.0          0.0 Partly Cloudy           83.6     Excellent
2026-10-02        26.2          67.0          0.0 Partly Cloudy           86.6     Excellent
2026-10-03        25.5          70.0          0.0        Cloudy           87.5     Excellent
2026-10-04        25.2          74.0          0.0        Cloudy           86.8     Excellent
2026-10-05        25.2          81.0          3.8         Rainy           78.3          Good
2026-10-06        27.1          73.0          0.0        Cloudy           81.5     Excellent
2026-10-07        25.9          65.0          0.0 Partly Cloudy           88.3     Excellent
2026-10-08        27.3          68.0          4.2         Rainy           76.6          Good


## Step 6 — NumPy statistics
`.to_numpy()` turns a pandas column into a NumPy array, so we can use NumPy's maths functions.

In [18]:
tmax = weather["temp_max_C"].to_numpy()

print("Mean of daily max temp :", round(np.mean(tmax), 2), "°C")
print("Median                 :", round(np.median(tmax), 2), "°C")
print("Standard deviation     :", round(np.std(tmax), 2), "°C")
print("Highest                :", np.max(tmax), "°C on", weather.loc[np.argmax(tmax), "date"].date())
print("Lowest daytime high    :", np.min(tmax), "°C on", weather.loc[np.argmin(tmax), "date"].date())
print("90th percentile        :", round(np.percentile(tmax, 90), 2), "°C")

corr_temp_hum = np.corrcoef(weather["temp_avg_C"], weather["humidity_pct"])[0, 1]
corr_temp_comf = np.corrcoef(weather["temp_avg_C"], weather["comfort_score"])[0, 1]
print("\nCorrelation  temperature vs humidity     :", round(corr_temp_hum, 3))
print("Correlation  temperature vs comfort score:", round(corr_temp_comf, 3))
print("(+1 = move together, -1 = move opposite, 0 = no link)")

Mean of daily max temp : 28.35 °C
Median                 : 28.6 °C
Standard deviation     : 2.95 °C
Highest                : 33.9 °C on 2026-10-08
Lowest daytime high    : 22.1 °C on 2026-11-11
90th percentile        : 32.65 °C

Correlation  temperature vs humidity     : 0.625
Correlation  temperature vs comfort score: 0.505
(+1 = move together, -1 = move opposite, 0 = no link)


> **Note:** `np.std()` divides by *n* (population standard deviation) while pandas `describe()` divides by *n − 1* (sample standard deviation), so the two values differ slightly (2.95 vs 2.98). Both are correct — they just answer slightly different questions.

## Step 7 — Records of the season
`idxmax()` and `idxmin()` give the row number of the biggest / smallest value.

In [19]:
def show(title, idx, col, unit):
    r = weather.loc[idx]
    print(f"{title:<22}: {r[col]} {unit} on {r['date'].date()} ({r['day_name']})")

show("Hottest day", weather["temp_max_C"].idxmax(), "temp_max_C", "°C")
show("Coldest night", weather["temp_min_C"].idxmin(), "temp_min_C", "°C")
show("Wettest day", weather["rainfall_mm"].idxmax(), "rainfall_mm", "mm")
show("Windiest day", weather["wind_kmph"].idxmax(), "wind_kmph", "km/h")
show("Most humid day", weather["humidity_pct"].idxmax(), "humidity_pct", "%")
show("Most comfortable day", weather["comfort_score"].idxmax(), "comfort_score", "points")
show("Least comfortable day", weather["comfort_score"].idxmin(), "comfort_score", "points")

cool = weather[weather["temp_min_C"] < 15]
if not cool.empty:
    print("\nFirst night below 15 °C :", cool["date"].iloc[0].date(), "| total such nights:", len(cool))
print("Total rainfall          :", round(weather["rainfall_mm"].sum(), 1), "mm on", int((weather["rainfall_mm"] > 0).sum()), "rainy days")

Hottest day           : 33.9 °C on 2026-10-08 (Thursday)
Coldest night         : 9.3 °C on 2026-11-13 (Friday)
Wettest day           : 15.9 mm on 2026-11-08 (Sunday)
Windiest day          : 13.6 km/h on 2026-10-18 (Sunday)
Most humid day        : 81.0 % on 2026-10-05 (Monday)
Most comfortable day  : 97.8 points on 2026-10-11 (Sunday)
Least comfortable day : 59.2 points on 2026-11-14 (Saturday)

First night below 15 °C : 2026-10-17 | total such nights: 23
Total rainfall          : 76.5 mm on 11 rainy days


## Step 8 — Group the data (weekly and monthly summary)
`groupby()` splits the table into groups (like all rows of week 1, week 2 …) and `agg()` calculates something for each group.

In [20]:
weekly = weather.groupby("week_no").agg(
    days=("date", "count"),
    avg_max_C=("temp_max_C", "mean"),
    avg_min_C=("temp_min_C", "mean"),
    avg_humidity=("humidity_pct", "mean"),
    total_rain_mm=("rainfall_mm", "sum"),
    avg_comfort=("comfort_score", "mean"),
).round(1)

print("WEEKLY SUMMARY (week 7 has only 4 days)")
print(weekly.to_string())

best_week = weekly["avg_comfort"].idxmax()
worst_week = weekly["avg_comfort"].idxmin()
print(f"\nBest week : {best_week} (comfort {weekly.loc[best_week, 'avg_comfort']})")
print(f"Worst week: {worst_week} (comfort {weekly.loc[worst_week, 'avg_comfort']})")

monthly = weather.groupby("month", sort=False).agg(
    avg_max_C=("temp_max_C", "mean"),
    avg_min_C=("temp_min_C", "mean"),
    total_rain_mm=("rainfall_mm", "sum"),
    avg_comfort=("comfort_score", "mean"),
).round(1)
print("\nMONTHLY SUMMARY")
print(monthly.to_string())

WEEKLY SUMMARY (week 7 has only 4 days)
         days  avg_max_C  avg_min_C  avg_humidity  total_rain_mm  avg_comfort
week_no                                                                      
1           7       31.8       20.3          71.0            3.8         84.7
2           7       31.5       19.3          63.6           10.8         88.1
3           7       28.6       15.7          63.1            0.0         89.1
4           7       28.9       15.0          60.4            3.9         88.8
5           7       26.7       13.4          56.3           14.9         81.0
6           7       24.8       11.7          55.1           23.6         76.0
7           4       24.4       10.8          54.8           19.5         71.6

Best week : 3 (comfort 89.1)
Worst week: 7 (comfort 71.6)

MONTHLY SUMMARY
          avg_max_C  avg_min_C  total_rain_mm  avg_comfort
month                                                     
October        29.9       17.2           23.7         86.9
Novem

## Step 9 — Count categories
`value_counts()` counts how many times each word appears in a column.

In [21]:
print("SKY CONDITION (days)")
print(weather["sky"].value_counts().to_string())

print("\nRAIN LEVEL (days)")
print(weather["rain_level"].value_counts().to_string())

print("\nCOMFORT LABEL (days)")
print(weather["comfort_label"].value_counts().to_string())

print("\nAVERAGE COMFORT BY WEEKDAY")
by_day = weather.groupby("day_name")["comfort_score"].mean().round(1).sort_values(ascending=False)
print(by_day.to_string())

SKY CONDITION (days)
sky
Partly Cloudy    24
Rainy            11
Sunny             7
Cloudy            4

RAIN LEVEL (days)
rain_level
Dry         35
Light        5
Moderate     5
Heavy        1

COMFORT LABEL (days)
comfort_label
Excellent    34
Good         10
Fair          2

AVERAGE COMFORT BY WEEKDAY
day_name
Monday       85.7
Sunday       84.7
Wednesday    84.4
Friday       84.2
Tuesday      82.1
Thursday     82.0
Saturday     81.5


## Step 10 — Smooth the trend with a 7-day moving average
Daily temperatures jump up and down. A **rolling mean** of the last 7 days shows the real direction.

In [22]:
weather["temp_max_7day"] = weather["temp_max_C"].rolling(window=7).mean().round(1)

drop = weather["temp_max_7day"].dropna().iloc[0] - weather["temp_max_7day"].iloc[-1]
print("7-day average of max temperature: first =", weather["temp_max_7day"].dropna().iloc[0],
      "°C, last =", weather["temp_max_7day"].iloc[-1], "°C")
print("Daytime highs fell by about", round(drop, 1), "°C over the season")

7-day average of max temperature: first = 31.8 °C, last = 24.6 °C
Daytime highs fell by about 7.2 °C over the season


## Step 11 — A small "day report" tool (function)
Type any date from the dataset and get a short report. Change the dates below and run again.

In [23]:
def day_report(date_text):
    row = weather[weather["date"] == pd.to_datetime(date_text)]
    if row.empty:
        print(f"No data for {date_text}. Choose a date between",
              weather["date"].min().date(), "and", weather["date"].max().date())
        return
    r = row.iloc[0]
    print("=" * 46)
    print(f" WEATHER REPORT: {r['date'].date()} ({r['day_name']})")
    print("=" * 46)
    print(f" Temperature : {r['temp_min_C']} to {r['temp_max_C']} °C (avg {r['temp_avg_C']})")
    print(f" Humidity    : {r['humidity_pct']} %")
    print(f" Rainfall    : {r['rainfall_mm']} mm ({r['rain_level']})")
    print(f" Wind        : {r['wind_kmph']} km/h")
    print(f" Sky         : {r['sky']}")
    print(f" Comfort     : {r['comfort_score']} / 100 -> {r['comfort_label']}")
    if r["my_note"]:
        print(f" My note     : {r['my_note']}")
    print("=" * 46)

day_report("2026-10-12")
day_report("2026-11-01")
day_report("2026-12-25")   # outside the data -> shows the friendly error message

 WEATHER REPORT: 2026-10-12 (Monday)
 Temperature : 19.2 to 33.0 °C (avg 26.1)
 Humidity    : 61.0 %
 Rainfall    : 0.0 mm (Dry)
 Wind        : 11.6 km/h
 Sky         : Partly Cloudy
 Comfort     : 89.3 / 100 -> Excellent
 My note     : Unit test week begins
 WEATHER REPORT: 2026-11-01 (Sunday)
 Temperature : 13.7 to 27.9 °C (avg 20.8)
 Humidity    : 41.0 %
 Rainfall    : 0.0 mm (Dry)
 Wind        : 9.8 km/h
 Sky         : Sunny
 Comfort     : 86.8 / 100 -> Excellent
 My note     : Weekend trip planned
No data for 2026-12-25. Choose a date between 2026-10-01 and 2026-11-15


## Step 12 — Save the final analysed data

In [24]:
weather.to_csv("ashish_weather_analysed.csv", index=False)
print("Saved: ashish_weather_analysed.csv with", weather.shape[0], "rows and", weather.shape[1], "columns")

# In Google Colab you can download the file by removing the # in the next two lines:
# from google.colab import files
# files.download("ashish_weather_analysed.csv")

Saved: ashish_weather_analysed.csv with 46 rows and 17 columns


## ✅ Conclusion
* Over the 46 days the weather changed from warm and humid to cool and drier air — daytime highs and night lows both fell clearly and humidity dropped. Rain stayed irregular: the wettest spell actually came in the last three weeks (weeks 5–7), which pulled the comfort score down.
* NumPy gave quick statistics (mean, median, standard deviation, percentile, correlation).
* pandas helped me clean missing values, create new columns, group by week and month, and count categories.
* My own **Ashish Comfort Score** turned four weather factors into one easy number.

### Ideas to extend this project
* Replace the generated data with real readings from a weather website or your own thermometer.
* Add a "feels-like" temperature or a UV index column.
* Make an input box with `input()` so that anyone can type a date and get the report.

*Made by Ashish Kumar (Admission No. 2026B15310080) — B.Tech 1st Year mini-project.*